In [ ]:
import pandas as pd
import re
from clean_unique_ids import clean_id

#------------------------------------------------------------------
# file paths
#------------------------------------------------------------------
# path to the GIS OneDrive folder where flu table is located
one_drive_path = "C:/Users/JKolberg/OneDrive - PSRC/GIS - Projects/FLU"

# input file paths
flu_path = f"{one_drive_path}/Zoning_2026_d3.xlsx"
sfmf_path = f"{one_drive_path}/SF_MF_may26.xlsx"

# output file paths
flu_output_path = f"Zoning_2026_d3.xlsx"
unmatched_output_path = f"unmatched_sf_mf.xlsx"


#------------------------------------------------------------------
# configurations
#------------------------------------------------------------------
# remove strings: list of strings to remove from the Juris_zn values for matching
remove_strings = ['cityof']

# manual matches 
# these are cases where the id cleaning process doesn't result in a match between the flu and
# sf/mf tables. The key is the juris_zn from the flu table, and the value is the Juris_zn
# from the sf/mf table.
manual_matches = {
    # juris_zn_flu: Juris_zn from sf/mf table
    'Auburn_DUC_FR': 'Auburn_DUC-RF',
    'Bainbridge_Island_HS1':'Bainbridge Island_HSR',
    'Bainbridge_Island_HS2':'Bainbridge Island_HSR',
    'Kenmore_CB_Juanita': 'Kenmore_CB',
    'Kenmore_CB_West': 'Kenmore_CB',
    'Kenmore_UC_East':'Kenmore_UC',
    'Kenmore_UC_West': 'Kenmore_UC',
    'Beaux_Arts_SFR':'Beaux Arts Village_General_Residential',
    'Monroe_DC':'Monroe_Downtown_Commercial_DC',
    'Monroe_LS':'Monroe_Limited_Open_Space_LS',
    'Monroe_MG':'Monroe_Mixed_Use__General_MG',
    'Monroe_MN':'Monroe_Mixed_Use__Neighborhood_MN',
    'Monroe_R15':'Monroe_Single-Family_Residential__15_Units_per_Acre_R15',
    'Monroe_R25':'Monroe_Multifamily_Residential__25_Units_per_Acre_R25',
    'Monroe_R7':'Monroe_Single-Family_Residential__7_Units_per_Acre_R7',
    'Normandy_Park_R-15': 'Normancy Park_R-15',
    'Seattle_MR':'Seattle_MR_RC',
    'Seattle_NC1-40':'Seattle_NC1P-40',
    'Seattle_NR': 'Seattle_NR1',
    'Shoreline_TC-1':'Shoreline_TC-1_TC-2_TC-3',
    'Shoreline_TC-2':'Shoreline_TC-1_TC-2_TC-3',
    'Shoreline_TC-3':'Shoreline_TC-1_TC-2_TC-3',
    'Shoreline_TC-4':'Shoreline_TC-1_TC-2_TC-3',
    'Snohomish_County_LDMRPRD':'Snohomish_PRD-LDMR',
    'Snohomish_County_R-7200':'Snohomish_R-7200',
    'Snohomish_County_R-7200PRD':'Snohomish_R-7200PRD'
}

In [ ]:
# read in the flu table
flu = (
    pd.read_excel(flu_path)
    # drop the sf/mf columns if they exist, since we'll be replacing them with the values from the matched table
    .drop(columns=['SingleFamily_Use', 'MultiFamily_Use', 'MinDU_lot'], errors='ignore')
)

# read in the sf/mf table and filter to rows with data in at least one of the three relevant columns
sfmf = (
    pd.read_excel(sfmf_path)
    .query('SingleFamily_Use.notna() | MultiFamily_Use.notna() | MinDU_lot.notna()')
)

In [32]:
# clean the Juris_zn column in the sf/mf table to create a new column for matching to the flu table
sfmf['juris_zn_cleaned'] = sfmf['Juris_zn'].apply(clean_id)

In [33]:
# number of zones in the sf/mf table that do not have a match in the flu table
num_no_match = len(sfmf[~sfmf['juris_zn_cleaned'].isin(flu['juris_zn'])])
print(f"Number of zones in the sf/mf table that do not have a match in the flu table: {num_no_match}")

Number of zones in the sf/mf table that do not have a match in the flu table: 54


In [ ]:
# clean up ids even further to try to get more matches
# remove all non-alphanumeric characters and convert to lowercase
def normalize(s):
    if pd.isna(s):
        return s
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

# remove custom list of strings
def custom_normalize(s, remove_strings):
    for remove_str in remove_strings:
        s = s.replace(remove_str, '')
    return s

sfmf['juris_zn_norm'] = sfmf['juris_zn_cleaned'].apply(normalize)
flu['juris_zn_norm'] = flu['juris_zn'].apply(normalize)
sfmf['juris_zn_norm'] = sfmf['juris_zn_norm'].apply(custom_normalize,remove_strings)
flu['juris_zn_norm'] = flu['juris_zn_norm'].apply(custom_normalize,remove_strings)

# join the tables to see which zones still do not have a match after cleaning up the ids
matched = sfmf.merge(
    flu[['juris_zn', 'juris_zn_norm']].rename(columns={'juris_zn': 'juris_zn_flu'}),
    on='juris_zn_norm',
    how='left',
)

# separate out the unmatched and matched zones
unmatched = matched[matched['juris_zn_flu'].isna()]
matched = matched[~matched['juris_zn_flu'].isna()]
print(f"Number of zones in the sf/mf table that still do not have a match in the flu table: {len(unmatched)}")

Number of zones in the sf/mf table that still do not have a match in the flu table: 26


In [ ]:
# build a dataframe of manual matches and join to sfmf to pull in the sf/mf values
manual_df = (
    pd.DataFrame(list(manual_matches.items()), columns=['juris_zn_flu', 'Juris_zn'])
    .merge(sfmf, on='Juris_zn', how='left')
)

In [36]:
# warn about any manual matches that didn't find a corresponding row in sfmf
missing_manual = manual_df[manual_df['SingleFamily_Use'].isna() & manual_df['MultiFamily_Use'].isna() & manual_df['MinDU_lot'].isna()]
if len(missing_manual) > 0:
    print("Manual matches with no corresponding sf/mf row:")
    print(missing_manual[['juris_zn_flu', 'Juris_zn']])

# append the manual matches to the matched dataframe so they get joined into the flu table
matched = pd.concat([matched, manual_df], ignore_index=True)

# remove any zones that were resolved via manual matches from the unmatched set
unmatched = unmatched[~unmatched['Juris_zn'].isin(manual_matches.values())]
print(f"Number of zones in the sf/mf table that still do not have a match after manual matches: {len(unmatched)}")

Number of zones in the sf/mf table that still do not have a match after manual matches: 6


In [37]:
# join matched zones back to the flu table with the relevant columns from the sf/mf table
out_flu = (flu.merge(
    matched[['juris_zn_flu', 'SingleFamily_Use', 'MultiFamily_Use', 'MinDU_lot']]
        .drop_duplicates(subset='juris_zn_flu'),
    left_on='juris_zn',
    right_on='juris_zn_flu',
    how='left',
    )
    .drop(columns=['juris_zn_flu','juris_zn_norm'])
)

In [ ]:
# write the new flu table with the sf/mf columns added
out_flu.to_excel(flu_output_path, index=False)
# write the unmatched zones to a separate file for review
unmatched.to_excel(unmatched_output_path, index=False)